# 02_customer_impact.ipynb
Objectives:
- Validate H2: SLA violations lead to worse customer experience and retention.
- Quantify how customer harm increases with delay severity (dose–response).
- Provide multi-level evidence:
  - Level 1: Descriptive signposts (on-time vs delayed).
  - Level 2: Stratified comparison (approx. matched analysis).
  - Level 3: Within-seller before/after analysis around severe events.
  - Level 4: Delay bucket dose–response.
- Prepare artifacts for downstream ROI and intervention simulations.

Notes:
- This notebook operates at the order-level and links SLA metrics to
  customer experience outcomes (reviews, cancellations, repeat purchases).
- It relies on the seller-level SLA pipeline already validated in
  `01_seller_risk.ipynb`.
- Economic impact / ROI will be handled in later modules.

### 0. Import & Settings

In [26]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import altair as alt
from IPython.core.interactiveshell import InteractiveShell
from pathlib import Path
import sys

pd.set_option("display.max_columns", 50)
InteractiveShell.ast_node_interactivity = "all"
alt.data_transformers.disable_max_rows()


NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))
    

# Import project modules
from config import DATA_INTERIM, DATA_PROCESSED
from features.seller_metrics import validate_orders_sellers
from features.customer_impact import (
    build_order_customer_panel,
    summarize_cx_by_sla_flag,
    summarize_cx_by_delay_bucket,
    summarize_cx_by_strata_and_sla,
    compute_stratified_deltas,
#     within_seller_before_after_summary,
)
from data.preprocessing import (
    load_orders_sellers,
)
from data.load_raw import (
    load_reviews,
    load_customers,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


DataTransformerRegistry.enable('default')

### 1. Data Load & Panel Construction

This section:
- Loads the base tables required for H2:
  - `orders_sellers`: wide table with SLA metrics at order + seller level.
  - `reviews`: order-level review scores.
  - `customers`: customer dimension (for repeat behavior and geography).
- Validates the SLA-wide table schema using `validate_orders_sellers`.
- Builds an **order-level panel** that links SLA and customer experience:
  - review_score / low-rating flags
  - cancellation flag
  - delay buckets
  - 30-day repeat purchase indicator

In [2]:
# load base tables
orders_sellers = load_orders_sellers()
reviews = load_reviews()
customers = load_customers()

# print basic attributes of the tables
print(
    f"orders_sellers shape: {orders_sellers.shape}, "
    f"reviews shape: {reviews.shape}, "
    f"customers shape: {customers.shape}"
)

# print the time range of the orders_sellers table and display the first few rows
print(
    "orders_sellers time range:",
    orders_sellers["order_purchase_timestamp"].min(),
    "to",
    orders_sellers["order_purchase_timestamp"].max(),
)
orders_sellers.head()

orders_sellers shape: (99441, 12), reviews shape: (99224, 7), customers shape: (99441, 5)
orders_sellers time range: 2016-09-04 21:15:19 to 2018-10-17 17:30:18


,order_id,seller_id,customer_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delay_days,is_sla_violation,is_severe_violation,has_time_anomaly,product_category_name_english
0,e481f51cbdc54678b7cc49136f2d6af7,3504c0cb71d7fa48d967e0e4c94d59d9,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,-8.0,False,False,False,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,289cdb325fb7e7f891c38608bf9e0962,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,-6.0,False,False,False,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,4869f7a5dfa277a7dca6462dcf3b52b2,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,-18.0,False,False,False,auto
3,949d5b44dbf5de918fe9c16f97b45f8a,66922902710d126a0e7d26b0e3805106,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,-13.0,False,False,False,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,2c9e548be18521d1c43cde1c582c6de8,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,-10.0,False,False,False,stationery


In [3]:
# validate the orders_sellers table, same as 01
validate_orders_sellers(orders_sellers, gmv_col = None)

orders_sellers contains unexpected order_status values: {'approved'}


In [4]:
# build order-level panel linking SLA and CX
panel = build_order_customer_panel(
    orders_sellers=orders_sellers,
    order_reviews=reviews,
    customers=customers,
    max_horizon_days=30,
    low_rating_threshold=3,
    very_low_rating_threshold=2,
)

panel.head()

,order_id,seller_id,customer_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delay_days,is_sla_violation,is_severe_violation,has_time_anomaly,product_category_name_english,review_score,review_creation_date,is_low_rating,is_very_low_rating,is_canceled,delay_bucket,customer_city,customer_state,next_order_time,days_to_next_order,repeat_within_horizon
68955,5f79b5b0931d63f1a42989eb65b9da6e,7aa4334be125fcdd2ba64b3180029f14,00012a2ce6f8dcda20d059ce98491703,delivered,2017-11-14 16:08:26,2017-11-28 15:41:30,2017-12-04,-6.0,False,False,False,toys,1.0,2017-11-29,True,True,False,on_time_or_early,osasco,SP,NaT,NaN,False
10070,a44895d095d7e0702b6a162fa2dbeced,2a1348e9addc1af5aaa619b1a3679d6b,000161a058600d5901f007fab4c27140,delivered,2017-07-16 09:40:32,2017-07-25 18:57:33,2017-08-04,-10.0,False,False,False,health_beauty,4.0,2017-07-26,False,False,False,on_time_or_early,itapecerica,MG,NaT,NaN,False
66244,316a104623542e4d75189bb372bc5f8d,46dc3b2cc0980fb8ec44634e21d2718e,0001fd6190edaaf884bcaf3d49edf079,delivered,2017-02-28 11:06:43,2017-03-06 08:57:49,2017-03-22,-16.0,False,False,False,baby,5.0,2017-03-07,False,False,False,on_time_or_early,nova venecia,ES,NaT,NaN,False
43391,5825ce2e88d5346438686b0bba99e5ee,aafe36600ce604f205b86b5084d3d767,0002414f95344307404f0ace7a26f1d5,delivered,2017-08-16 13:09:20,2017-09-13 20:06:02,2017-09-14,-1.0,False,False,False,cool_stuff,5.0,2017-09-14,False,False,False,on_time_or_early,mendonca,MG,NaT,NaN,False
5916,0ab7fb08086d4af9141453c91878ed7a,4a3ca9315b744ce9f8e9374361493884,000379cdec625522490c315e70c7a9fb,delivered,2018-04-02 13:42:17,2018-04-13 20:21:08,2018-04-18,-5.0,False,False,False,bed_bath_table,4.0,2018-04-14,False,False,False,on_time_or_early,sao paulo,SP,NaT,NaN,False


In [5]:
# Basic sanity checks on the panel
panel[["review_score", "is_low_rating", "is_canceled", "repeat_within_horizon"]].isna().mean()

review_score             0.007681
is_low_rating            0.000000
is_canceled              0.000000
repeat_within_horizon    0.000000
dtype: float64

In [6]:
# Delay buckets distribution
panel["delay_bucket"].value_counts(dropna=False).sort_index()

delay_bucket
on_time_or_early    90442
1-2_days_late        1374
3-5_days_late        1403
6+_days_late         3786
NaN                  2987
Name: count, dtype: int64

In [7]:
# Repeat within horizon distribution
panel["repeat_within_horizon"].mean()

np.float64(0.005510440835266821)

### 2. Level 1 – Descriptive Signposts (On-time vs Delayed)

**Business interpretation**
- We first compare customer experience metrics between **on-time** and
  **delayed** orders, without any matching or stratification.
- This provides the most basic evidence for H2:
  
  Are SLA violations associated with worse reviews, higher cancellations,and lower repeat rates?

We will:
- Compare CX metrics by:
  - `is_sla_violation` (any SLA violation).
  - `is_severe_violation` (severe violations only).
- Visualize low-rating, cancellation, and repeat rates for violation vs non-violation groups.

In [8]:
# SLA violations vs CX
level1_sla = summarize_cx_by_sla_flag(panel, sla_flag_col="is_sla_violation")
level1_sla

,sla_flag,orders,mean_review,low_rating_rate,very_low_rating_rate,cancel_rate,repeat_rate
0,0,92906,4.211789,0.193259,0.113188,0.006722,0.005598
1,1,6535,2.271139,0.715831,0.609477,0.000152,0.004266


In [9]:
# Severe SLA violations vs CX
level1_severe = summarize_cx_by_sla_flag(panel, sla_flag_col="is_severe_violation")
level1_severe

,sla_flag,orders,mean_review,low_rating_rate,very_low_rating_rate,cancel_rate,repeat_rate
0,0,96578,4.155663,0.208891,0.127274,0.006467,0.005509
1,1,2863,1.700143,0.857242,0.769364,0.000347,0.005557


In [10]:
# Prepare data for visualization: low rating, very low rating, cancellation, repeat rate
level1_long = (
    level1_sla.melt(
        id_vars=["sla_flag", "orders"],
        value_vars=[
            "low_rating_rate",
            "very_low_rating_rate",
            "cancel_rate",
            "repeat_rate",
        ],
        var_name="metric",
        value_name="value",
    )
    .assign(
        metric=lambda d: d["metric"].map(
            {
                "low_rating_rate": "Low rating (≤3)",
                "very_low_rating_rate": "Very low rating (≤2)",
                "cancel_rate": "Cancellation rate",
                "repeat_rate": "30d repeat rate",
            }
        )
    )
)

base = alt.Chart(level1_long).encode(
    x=alt.X(
        "sla_flag:N",
        title="SLA violation flag (0 = on-time, 1 = violation)",
        axis=alt.Axis(labelAngle=0),
    ),
    y=alt.Y("value:Q", title="Rate"),
    color=alt.Color("metric:N", title="Metric"),
)

bars = base.mark_bar().encode(
    xOffset="metric:N",
)

(bars).properties(
    title="Customer experience by SLA violation flag",
    width=280,
    height=280,
)

alt.Chart(...)

### 3. Level 4 – Dose–Response: Delay Bucket vs CX

**Business interpretation**
- We move from a binary SLA flag to **continuous delay severity**, bucketed into:
  - on-time or early
  - 1–2 days late
  - 3–5 days late
  - 6+ days late
- We expect:
  - Mean review score ↓ as delay increases.
  - Low-rating and cancellation rates ↑ with delay.
  - 30-day repeat purchase probability ↓ with delay.

This provides **dose–response evidence** for H2.


In [11]:
# Convert rates to percentages for human-readable tables
dose_summary = summarize_cx_by_delay_bucket(panel)
dose_display = dose_summary.copy()
for col in ["low_rating_rate", "cancel_rate", "repeat_rate"]:
    dose_display[col] = dose_display[col] * 100.0

dose_display

D:\NA_Work\projects\Marketplace-Sla-Risk-Analysis\src\features\customer_impact.py:218: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grp = panel.groupby("delay_bucket")


,delay_bucket,orders,mean_review,low_rating_rate,cancel_rate,repeat_rate
0,on_time_or_early,89941,4.289842,17.264103,0.005528,0.553946
1,1-2_days_late,1370,3.511029,40.320233,0.000000,0.291121
2,3-5_days_late,1400,2.467495,66.928011,0.000000,0.213828
3,6+_days_late,3765,1.740016,84.653988,0.026413,0.554675


In [12]:
# Mean review score vs delay bucket
chart_review = (
    alt.Chart(dose_summary)
    .mark_line(point=True)
    .encode(
        x=alt.X("delay_bucket:N", title="Delay bucket", sort=None, axis=alt.Axis(labelAngle=0)),
        y=alt.Y("mean_review:Q", title="Mean review score"),
    )
    .properties(
        title="Dose-response: mean review score vs delay severity",
        width=420,
        height=280,
    )
)
chart_review

alt.Chart(...)

In [13]:
# Low rating, cancellation, and repeat rate vs delay bucket
dose_long = (
    dose_summary.melt(
        id_vars=["delay_bucket", "orders"],
        value_vars=["low_rating_rate", "cancel_rate", "repeat_rate"],
        var_name="metric",
        value_name="value",
    )
    .assign(
        metric=lambda d: d["metric"].map(
            {
                "low_rating_rate": "Low rating (≤3)",
                "cancel_rate": "Cancellation rate",
                "repeat_rate": "30d repeat rate",
            }
        )
    )
)

chart_dose = (
    alt.Chart(dose_long)
    .mark_line(point=True)
    .encode(
        x=alt.X("delay_bucket:N", title="Delay bucket", sort=None, axis=alt.Axis(labelAngle=0)),
        y=alt.Y("value:Q", title="Rate"),
        color=alt.Color("metric:N", title="Metric"),
    )
    .properties(
        title="Dose-response: CX metrics vs delay severity",
        width=460,
        height=300,
    )
)
chart_dose

alt.Chart(...)

### 4. Level 2 – Stratified Comparison (Approx. Matched Analysis)

**Business interpretation**

- To move closer to causality, we compare on-time vs delayed orders within
  similar contexts, for example:
  - Same product category.
  - Same customer state.
- This controls for some confounding factors without building a full-blown
  propensity score model.

In this implementation, we:

- Run stratified analysis **one dimension at a time** to avoid over-granularity:
  - e.g., stratify by `product_category_name` alone, then by `customer_state` alone.
- For each stratum (e.g., each category), we compute:
  - Low-rating rate and 30d repeat rate for:
    - on-time orders,
    - delayed orders.
  - Deltas: `(violation – on_time)`.

This makes it easy to answer questions like:
- “Within a given category, how much worse are delayed orders compared to on-time ones?”
- “Which categories or regions are most sensitive to delays?”

In [14]:
# Identify available strata columns in the panel
candidate_strata_cols = []
if "product_category_name_english" in panel.columns:
    candidate_strata_cols.append("product_category_name_english")
if "customer_state" in panel.columns:
    candidate_strata_cols.append("customer_state")

candidate_strata_cols

['product_category_name_english', 'customer_state']

In [27]:
# Run one dimension at a time to avoid (category × state) over-granularity
for strata_col in candidate_strata_cols:
    print(f"\n=== Stratified by: {strata_col} ===")
    result = compute_stratified_deltas(panel, strata_col, min_orders=30)
    if result is not None:
        print(f"  Strata count: {len(result)}")
        display(result.sort_values("delta_repeat_rate").head(10))


=== Stratified by: product_category_name_english ===
  Strata count: 29


,product_category_name_english,low_rating_rate_on_time,low_rating_rate_violation,repeat_rate_on_time,repeat_rate_violation,delta_low_rating_rate,delta_repeat_rate
16,home_appliances,0.184655,0.606061,0.049415,0.000000,0.421405,-0.049415
17,home_confort,0.274052,0.771429,0.008746,0.000000,0.497376,-0.008746
12,furniture_decor,0.215256,0.711409,0.009388,0.002237,0.496154,-0.007151
24,sports_leisure,0.163051,0.733333,0.007047,0.000000,0.570282,-0.007047
18,housewares,0.184704,0.716612,0.003968,0.000000,0.531908,-0.003968
5,computers_accessories,0.208294,0.720764,0.008103,0.004773,0.512470,-0.003330
0,audio,0.238562,0.853659,0.003268,0.000000,0.615096,-0.003268
2,baby,0.194201,0.761062,0.003052,0.000000,0.566861,-0.003052
8,cool_stuff,0.169850,0.712919,0.002645,0.000000,0.543069,-0.002645
7,construction_tools_construction,0.186317,0.759259,0.001456,0.000000,0.572942,-0.001456



=== Stratified by: customer_state ===
  Strata count: 21


,customer_state,low_rating_rate_on_time,low_rating_rate_violation,repeat_rate_on_time,repeat_rate_violation,delta_low_rating_rate,delta_repeat_rate
8,MS,0.162614,0.750000,0.016717,0.000000,0.587386,-0.016717
3,DF,0.195397,0.796610,0.009794,0.000000,0.601213,-0.009794
12,PE,0.191391,0.758170,0.007285,0.000000,0.566779,-0.007285
10,PA,0.234286,0.764151,0.006857,0.000000,0.529865,-0.006857
6,MA,0.228435,0.744000,0.006390,0.000000,0.515565,-0.006390
4,ES,0.188081,0.644860,0.005467,0.000000,0.456779,-0.005467
16,RN,0.155756,0.863636,0.004515,0.000000,0.707880,-0.004515
20,SP,0.184126,0.643445,0.005356,0.001646,0.459319,-0.003710
9,MT,0.192532,0.698113,0.003501,0.000000,0.505581,-0.003501
7,MG,0.192597,0.691571,0.006170,0.003831,0.498974,-0.002338
